week 11 - in depth spacy

In [1]:
#preparing the enviorment
import pandas as pd
import spacy
from spacy.matcher import Matcher, PhraseMatcher

In [2]:
#uploading and labeling the nlp, matcher, and phrase matcher functions.
nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)
phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

Creating an empty data frame named result_list

In [3]:
results_list=[]

Defining the on_match_callback function. Matcher refers to the object running, through the doc (tokens), i is the index on the list of matches, and matches is the full list of matches.

Spacy stores labels as integers to save memory space.
span is an object created which is the slice of text that is identified as a match.

The results are appended to a temporary list with two columns, one of the text id and one of the actual matched text.

In [4]:
def on_match_callback(matcher, doc, i, matches):
    match_id, start, end = matches[i]
    string_id = nlp.vocab.strings[match_id]
    span = doc[start:end]

    results_list.append({
    "Matched Text": span.text,
    "Identifier": string_id
    })

The twitter sample data is uploaded.

In [5]:
df = pd.read_csv("twitter_sample.csv")

In [6]:
#a data frame for the final output is created.
final_output = []


This is the pattern for the Spacy Matcher, it is looking for an adjective modifying the noun rights, this can include civil right, human rights... or just plain rights because the adj is optional.

In [7]:
pattern_pos = [
    {"POS": "ADJ", "OP": "?"}, # Optional Adjective
    {"LOWER": "rights"}        # Followed by the word 'rights'
]

This cell defines the regular expression pattern (regex) for the matcher. it allows to catch variations of the word root without defining every possible version manually.
regex = Spacy evaluates the text using the regex engine.
?i = makes the entire expression case - insensitive.
matches any token that starts with the word equal, and the star includes any characters following the word equal.


In [8]:
pattern_regex = [
    {"TEXT": {"REGEX": r"(?i)^equal.*"}}
]

The patterns are added to the matcher object, which then identifies the text.

In [9]:
matcher.add("SOCIAL_CAUSE", [pattern_pos, pattern_regex], on_match=on_match_callback)

phrase matcher is used to find a specific list of words, and not a linguistic pattern. a few common terms for social causes are put in, so the phrase_matcher can identify them.

In [10]:
terms = ["climate change", "poverty relief", "end hunger", "black lives matter"]
phrase_patterns = [nlp.make_doc(text) for text in terms]

phrase_matcher.add("SOCIAL_CAUSE", phrase_patterns, on_match=on_match_callback)


this cell contains the core execution logic of the NLP pipeline. it iterates through the dataset to apply the matching rules and structure the findings. i put a limit of 100 doc. because the computer was freezing from the overload.

In [11]:
for index, row in df.iterrows():
    results_list = []
    if index > 100: 
        break

    doc = nlp(str(row['Tweet Content']))

    matcher(doc)
    phrase_matcher(doc)


    for item in results_list:
        final_output.append({
            "Name":row['Name'],
            "Matched Text": item["Matched Text"],
            "Identifier": item["Identifier"]
        })

result_df = pd.DataFrame(final_output)

I printed the results, they were empty. which means there were no matches in the first 100 doc, so i printed a drop of the data to see the common social causes expressed.

In [12]:
print(result_df)

Empty DataFrame
Columns: []
Index: []


In [13]:
#printing a few
for tweet in df['Tweet Content'].head(10):
    print(f"--- Tweet: {tweet}")

--- Tweet: Pets change our lives &amp; become a part of our families ❤️
That's why our members offer many solutions to help you to enjoy a long-lasting bond with your happy &amp; healthy pet 🐱🐶
#MorethanMedicine #PetCare #PetsareFamily https://t.co/fZNIXge9a3
--- Tweet: Another spot of our #morethanmedicine bus in #bristol this week! If you need support with your cancer diagnosis call us on 0303 3000 118. #livingwellwithcancer https://t.co/eZGLz0BkXB
--- Tweet: What a great team ⁦@HealthSourceOH⁩ ⁦@Local12⁩ #morethanmedicine https://t.co/g2YzMDUpVA
--- Tweet: What a great team ⁦@HealthSourceOH⁩ ⁦@Local12⁩ #morethanmedicine https://t.co/g2YzMDUpVA
--- Tweet: What a great team ⁦@HealthSourceOH⁩ ⁦@Local12⁩ #morethanmedicine https://t.co/g2YzMDUpVA
--- Tweet: What a great team ⁦@HealthSourceOH⁩ ⁦@Local12⁩ #morethanmedicine https://t.co/g2YzMDUpVA
--- Tweet: Will you be at #FIX19? Want a preview of @AG_EM33 story? Then check back Monday for #ChangeOfHeart where we sat down with Alin to disc

I changed the pattern attributes for the matcher to an animal followed by a noun(optional).

In [14]:
#Token Attributes (POS)
pattern_attr = [
    {"LOWER": "animal"}, 
    {"POS": "NOUN"}
]

I modified the phrase matcher with new phrases that were more similar to the kinds of issues in the data.

In [15]:
#PhraseMatcher
terms = ["living well with cancer", "climate change", "social prescribing"]
phrase_patterns = [nlp.make_doc(text) for text in terms]

I reexecuted the new versions.

In [16]:
matcher.add("SOCIAL_CAUSE", [pattern_attr, pattern_regex], on_match=on_match_callback)
phrase_matcher.add("SOCIAL_CAUSE", phrase_patterns, on_match=on_match_callback)

for index, row in df.iterrows():
    results_list = []  
    
    # Process the text
    doc = nlp(str(row['Tweet Content']))
    
    # Execute Matchers
    matcher(doc)
    phrase_matcher(doc)
    
    # Transfer from local collector to final output
    for item in results_list:
        final_output.append({
            "Name": row['Name'],
            "Matched Text": item["Matched Text"],
            "Identifier": item["Identifier"]
        })

# --- 5. RESULT ---
result_df = pd.DataFrame(final_output)

In [17]:
#this time the results worked very well.
print(result_df)

                          Name        Matched Text    Identifier
0                 Alison Bevan  social prescribing  SOCIAL_CAUSE
1            Cyton Biosciences       animal health  SOCIAL_CAUSE
2                   hany shita       Animal health  SOCIAL_CAUSE
3                   hany shita       animal health  SOCIAL_CAUSE
4           Alexandre Carvalho      animal welfare  SOCIAL_CAUSE
5                   hany shita       Animal health  SOCIAL_CAUSE
6           AnimalhealthEurope       Animal health  SOCIAL_CAUSE
7           Masheti Collince B       animal health  SOCIAL_CAUSE
8                        pauli       animal health  SOCIAL_CAUSE
9             kristof van hoye      animal welfare  SOCIAL_CAUSE
10             Julie Vermooten      animal welfare  SOCIAL_CAUSE
11             Hester de Voogd      animal welfare  SOCIAL_CAUSE
12                  hany shita      animal welfare  SOCIAL_CAUSE
13  Colegio Médico Veterinario      animal welfare  SOCIAL_CAUSE
14               Roxane F

The following is  a clear summary of the results on this dataset.
The data frame is printed in a pleasing to the eye table, with the name, slice of matched text and the label social cause.

In [18]:
# Verify results without rerunning the main loop
print(f"Extraction Summary:")
print(f"- Total matches found in sample: {len(result_df)}")
print(f"- Number of unique names processed: {result_df['Name'].nunique()}")

# Show the most frequent matches found by the rules
print("\nTop Identified Social Causes:")
print(result_df['Matched Text'].value_counts())

# Display the structure of the resulting data
result_df.head()

Extraction Summary:
- Total matches found in sample: 60
- Number of unique names processed: 37

Top Identified Social Causes:
Matched Text
social prescribing    19
animal health         17
animal welfare        12
Animal medicines       8
Animal health          3
Social Prescribing     1
Name: count, dtype: int64


,Name,Matched Text,Identifier
0,Alison Bevan,social prescribing,SOCIAL_CAUSE
1,Cyton Biosciences,animal health,SOCIAL_CAUSE
2,hany shita,Animal health,SOCIAL_CAUSE
3,hany shita,animal health,SOCIAL_CAUSE
4,Alexandre Carvalho,animal welfare,SOCIAL_CAUSE
